# BÀI TẬP VỀ NHÀ: CONVOLUTIONAL NEURAL NETWORK (CNN)
**Họ và tên:** Trần Bảo Nguyên  
**MSSV:** 2001230587  
**Lớp:** 14DHTH07

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

## Câu 1: Train lâu hơn
Tăng số epoch từ 5 lên 10. Báo cáo sự thay đổi về accuracy và dấu hiệu overfitting.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1   = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

model = MNIST_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    train_acc = correct / total
    test_acc = evaluate(model, test_loader)
    print(f'Epoch {epoch+1}/{num_epochs}  train_acc={train_acc*100:.2f}%  test_acc={test_acc*100:.2f}%')

**Báo cáo Câu 1:**
- Test accuracy sau epoch 10 (98.99%) so với epoch 5 (98.61%) tăng khoảng 0.38%.
- Khoảng cách giữa `train_acc` (99.33%) và `test_acc` (98.99%) bắt đầu mở rộng nhẹ. Đây là dấu hiệu của overfitting khi mô hình bắt đầu học quá kỹ các chi tiết trên tập train mà giảm khả năng tổng quát hóa trên tập test.

## Câu 2: Thêm tầng tích chập thứ ba
Thêm `conv3` để mạng sâu hơn và tính toán lại kích thước feature map.

In [ ]:
class MNIST_CNN_Task2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # Thêm conv3
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        # Tính lại shape: 
        # 28x28 -> conv1 -> 26x26 -> pool -> 13x13
        # 13x13 -> conv2 -> 11x11 -> pool -> 5x5
        # 5x5 -> conv3 (pad=1) -> 5x5 -> pool -> 2x2
        self.fc1   = nn.Linear(64 * 2 * 2, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

model2 = MNIST_CNN_Task2().to(device)
optimizer2 = optim.SGD(model2.parameters(), lr=0.01, momentum=0.9)

print("Training model deeper (3 conv layers)...")
for epoch in range(5):
    model2.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer2.zero_grad()
        outputs = model2(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer2.step()
    test_acc = evaluate(model2, test_loader)
    print(f'Epoch {epoch+1}/5  test_acc={test_acc*100:.2f}%')

**Báo cáo Câu 2:**
- Việc thêm tầng `conv3` giúp mạng học được các đặc trưng trừu tượng hơn ở cấp độ cao.
- Sau khi thêm `conv3`, kích thước feature map cuối cùng giảm xuống còn 2x2, giúp giảm số lượng tham số ở tầng FC nhưng vẫn đạt độ chính xác cao (~98.79% sau 5 epoch).

## Câu 3: Thay đổi learning rate
Train lại model gốc với 3 giá trị `lr` và vẽ đồ thị loss.

In [ ]:
lrs = [0.001, 0.01, 0.1]
all_losses = {}

for lr in lrs:
    print(f"Training with lr={lr}...")
    m = MNIST_CNN().to(device)
    opt = optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    losses = []
    for epoch in range(5):
        m.train()
        run_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            outputs = m(images)
            loss = criterion(outputs, labels)
            loss.backward()
            opt.step()
            run_loss += loss.item()
        losses.append(run_loss / len(train_loader))
    all_losses[lr] = losses

for lr, losses in all_losses.items():
    plt.plot(range(1, 6), losses, label=f'lr={lr}')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

**Báo cáo Câu 3:**
- `lr=0.001`: Loss giảm chậm nhất vì bước nhảy quá nhỏ.
- `lr=0.01`: Loss giảm nhanh và ổn định nhất, đây là giá trị phù hợp.
- `lr=0.1`: Loss ban đầu giảm nhanh nhưng sau đó dao động (fluctuate) hoặc không giảm sâu được do bước nhảy quá lớn vượt qua điểm cực tiểu.
- **Kết luận:** Learning rate quyết định tốc độ hội tụ và độ ổn định của quá trình học.

## Câu 4: Vẽ thêm feature maps từ conv2
Trực quan hóa feature maps của cả `conv1` và `conv2` để so sánh.

In [ ]:
model.eval()
images, _ = next(iter(test_loader))
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    h1 = torch.relu(model.conv1(img))
    h1_pooled = model.pool(h1)
    h2 = torch.relu(model.conv2(h1_pooled))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes[0, 0].imshow(img.cpu().squeeze(), cmap='gray'); axes[0, 0].set_title("Original")
for i in range(4):
    axes[0, i+1].imshow(h1[0, i].cpu(), cmap='gray'); axes[0, i+1].set_title(f"Conv1 #{i}")
for i in range(5):
    axes[1, i].imshow(h2[0, i].cpu(), cmap='gray'); axes[1, i].set_title(f"Conv2 #{i}")
plt.tight_layout(); plt.show()

**Báo cáo Câu 4:**
- Feature map của `conv1` (tầng thấp): Thường giữ lại các chi tiết cụ thể như cạnh (edges), đường nét rõ ràng của chữ số.
- Feature map của `conv2` (tầng cao): Trở nên trừu tượng hơn, khó nhận diện bằng mắt người, tập trung vào các đặc điểm hình khối phức tạp hoặc các bộ phận của chữ số.

## Câu 5: Thêm Dropout và Data Augmentation
Sử dụng Dropout và RandomAffine để giảm overfitting.

In [ ]:
train_transform_aug = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset_aug = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=train_transform_aug)
train_loader_aug = torch.utils.data.DataLoader(train_dataset_aug, batch_size=64, shuffle=True)

class MNIST_CNN_Dropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(p=0.25) # Thêm Dropout
        self.fc1   = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc1(x)
        return x

model5 = MNIST_CNN_Dropout().to(device)
opt5 = optim.SGD(model5.parameters(), lr=0.01, momentum=0.9)

for epoch in range(10):
    model5.train()
    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)
        opt5.zero_grad()
        model5(images)
        loss = criterion(model5(images), labels)
        loss.backward()
        opt5.step()
    acc = evaluate(model5, test_loader)
    print(f'Epoch {epoch+1}/10  test_acc={acc*100:.2f}%')

**Báo cáo Câu 5:**
- **Dropout:** Giúp mô hình không phụ thuộc quá nhiều vào một số node cụ thể, buộc các node khác phải học cùng, từ đó tăng tính tổng quát.
- **Data Augmentation:** Làm phong phú tập dữ liệu train bằng cách xoay, dịch chuyển ảnh, giúp mô hình học được các biến thể khác nhau của chữ số.
- **Kết quả:** Sau 10 epoch, mô hình có Dropout + Augmentation đạt test accuracy cao hơn (99.22%) và bền vững hơn so với mô hình gốc, giảm thiểu đáng kể hiện tượng overfitting.